# 🤖 NewsBot Intelligence Engine 2.0
### ITAI 2373 — Final Project | Leroy Brown | Southern Shade Technologies

**Building on Midterm NewsBot (97.6% accuracy, 8 Modules)**  
This final version adds four advanced modules: topic modeling, text generation/summarization, multilingual intelligence, and a conversational interface.

---

| Module | Description |
|--------|-------------|
| M1–M8 | Midterm foundation (imported) |
| **M9** | Advanced Content Analysis — LDA topic modeling + enhanced sentiment |
| **M10** | Language Understanding & Generation — Summarization + Semantic Search |
| **M11** | Multilingual Intelligence — Language detection + translation |
| **M12** | Conversational Interface — NL query engine over the full pipeline |

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
# Run once. Colab: Runtime > Run all

!pip install -q \
    nltk spacy scikit-learn pandas numpy matplotlib seaborn \
    gradio kagglehub \
    transformers torch sentencepiece \
    sentence-transformers \
    langdetect deep-translator \
    gensim pyLDAvis \
    vaderSentiment

!python -m spacy download en_core_web_sm -q

import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'vader_lexicon']:
    nltk.download(pkg, quiet=True)

print('✅ All dependencies installed.')

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================
import warnings, re, json, textwrap
warnings.filterwarnings('ignore')

# Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter, defaultdict

# NLP — baseline
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import LatentDirichletAllocation

# M10 — Transformers
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util

# M11 — Multilingual
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
DetectorFactory.seed = 42

# M12 — Conversational
import gradio as gr

# Viz
import pyLDAvis
import pyLDAvis.lda_model

nlp = spacy.load('en_core_web_sm')
lemmatizer = WordNetLemmatizer()
sid = SentimentIntensityAnalyzer()
STOP = set(stopwords.words('english'))

CATS = ['business', 'entertainment', 'politics', 'sport', 'tech']
PALETTE = {'business':'#1f77b4','entertainment':'#e377c2',
           'politics':'#d62728','sport':'#2ca02c','tech':'#ff7f0e'}

print('✅ Imports complete.')

In [ ]:
# ============================================================
# CELL 3 — LOAD BBC DATASET  (same as midterm)
# ============================================================
import kagglehub, os, glob

path = kagglehub.dataset_download('hgultekin/bbcnewsarchive')
csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
print('Found CSVs:', csv_files)

df_raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df_raw.columns = df_raw.columns.str.lower().str.strip()

# Normalise column names
if 'category' not in df_raw.columns:
    for col in df_raw.columns:
        if df_raw[col].nunique() <= 10:
            df_raw.rename(columns={col: 'category'}, inplace=True)
            break
text_col = [c for c in df_raw.columns if 'text' in c or 'content' in c or 'article' in c][0]
df_raw.rename(columns={text_col: 'text'}, inplace=True)

df = df_raw[['category','text']].dropna()
df['category'] = df['category'].str.lower().str.strip()
df = df[df['category'].isin(CATS)]
df = df.sample(n=min(2000, len(df)), random_state=42).reset_index(drop=True)

print(f'\n✅ Dataset: {len(df)} articles')
print(df['category'].value_counts())

In [ ]:
# ============================================================
# CELL 4 — PREPROCESSING  (M2 baseline, enhanced)
# ============================================================
def preprocess(text, return_tokens=False):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP and len(t) > 2]
    return tokens if return_tokens else ' '.join(tokens)

df['clean'] = df['text'].apply(preprocess)
df['tokens'] = df['text'].apply(lambda x: preprocess(x, return_tokens=True))
df['word_count'] = df['tokens'].apply(len)

print('✅ Preprocessing complete')
print(df[['category','word_count']].groupby('category').mean().round(1))

In [ ]:
# ============================================================
# CELL 5 — M7 CLASSIFICATION  (midterm best model, re-trained)
# ============================================================
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = tfidf.fit_transform(df['clean'])
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'✅ Logistic Regression Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=CATS))

In [ ]:
# ============================================================
# CELL 6 — MODULE 9: ADVANCED CONTENT ANALYSIS
# Topic Modeling with LDA + Enhanced Sentiment
# ============================================================
print('=' * 60)
print('MODULE 9 — ADVANCED CONTENT ANALYSIS')
print('=' * 60)

# --- 9A: LDA TOPIC MODELING ---
print('\n[9A] LDA Topic Modeling...')

count_vec = CountVectorizer(max_df=0.9, min_df=5, max_features=5000)
dtm = count_vec.fit_transform(df['clean'])

N_TOPICS = 10
lda = LatentDirichletAllocation(
    n_components=N_TOPICS, random_state=42,
    learning_method='online', max_iter=20)
lda.fit(dtm)

vocab = count_vec.get_feature_names_out()

print('\n📌 Top 8 words per LDA topic:')
for i, comp in enumerate(lda.components_):
    top_words = [vocab[j] for j in comp.argsort()[:-9:-1]]
    print(f'  Topic {i+1:2d}: {" | ".join(top_words)}')

# Assign dominant topic per article
topic_dist = lda.transform(dtm)
df['dominant_topic'] = topic_dist.argmax(axis=1) + 1
df['topic_confidence'] = topic_dist.max(axis=1)

# --- 9B: ENHANCED MULTI-DIMENSION SENTIMENT ---
print('\n[9B] Enhanced Sentiment Analysis...')

def full_sentiment(text):
    scores = sid.polarity_scores(str(text))
    label = ('positive' if scores['compound'] >= 0.05
             else 'negative' if scores['compound'] <= -0.05
             else 'neutral')
    return pd.Series({
        'sent_compound': scores['compound'],
        'sent_pos': scores['pos'],
        'sent_neg': scores['neg'],
        'sent_neu': scores['neu'],
        'sent_label': label
    })

sent_df = df['text'].apply(full_sentiment)
df = pd.concat([df, sent_df], axis=1)

# --- VIZ: Topic distribution by category ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

topic_cat = df.groupby(['category','dominant_topic']).size().unstack(fill_value=0)
topic_cat.plot(kind='bar', stacked=True, ax=axes[0],
               colormap='tab20', legend=False)
axes[0].set_title('Topic Distribution by Category', fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Article Count')
axes[0].tick_params(axis='x', rotation=30)

# Enhanced sentiment by category
sent_means = df.groupby('category')[['sent_pos','sent_neg','sent_neu']].mean()
sent_means.plot(kind='bar', ax=axes[1],
                color=['#2ca02c','#d62728','#aec7e8'])
axes[1].set_title('Sentiment Dimensions by Category', fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Mean Score')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(['Positive','Negative','Neutral'])

plt.tight_layout()
plt.savefig('m9_advanced_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Module 9 complete — LDA topics extracted, enhanced sentiment scored')

In [ ]:
# ============================================================
# CELL 7 — MODULE 9C: pyLDAvis Interactive Topic Explorer
# ============================================================
print('[9C] Rendering interactive LDA visualization...')

pyLDAvis.enable_notebook()
vis = pyLDAvis.lda_model.prepare(lda, dtm, count_vec, sort_topics=False)
pyLDAvis.display(vis)

In [ ]:
# ============================================================
# CELL 8 — MODULE 10: LANGUAGE UNDERSTANDING & GENERATION
# Abstractive Summarization + Semantic Search
# ============================================================
print('=' * 60)
print('MODULE 10 — LANGUAGE UNDERSTANDING & GENERATION')
print('=' * 60)

# --- 10A: ABSTRACTIVE SUMMARIZATION ---
print('\n[10A] Loading DistilBART summarizer (CPU-friendly)...')

summarizer = hf_pipeline(
    'summarization',
    model='sshleifer/distilbart-cnn-12-6',
    device=-1  # CPU; change to 0 for GPU
)

def safe_summarize(text, max_len=130, min_len=30):
    """Summarize with safety truncation for model input limits."""
    tokens = text.split()
    if len(tokens) < 60:
        return text  # already short
    truncated = ' '.join(tokens[:512])
    try:
        result = summarizer(truncated, max_length=max_len,
                            min_length=min_len, do_sample=False)
        return result[0]['summary_text'].strip()
    except Exception as e:
        return f'[Summarization error: {e}]'

print('\n📝 Summarization Demo (3 articles per category):')
sample_rows = df.groupby('category').apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)

summaries_demo = []
for _, row in sample_rows.iterrows():
    summary = safe_summarize(row['text'])
    original_words = len(row['text'].split())
    summary_words = len(summary.split())
    compression = round((1 - summary_words/max(original_words,1)) * 100, 1)
    print(f'\n  [{row["category"].upper()}]')
    print(f'  Original: {original_words} words → Summary: {summary_words} words ({compression}% compression)')
    print(f'  Summary: {summary[:200]}...')
    summaries_demo.append({'category': row['category'],
                           'original_words': original_words,
                           'summary_words': summary_words,
                           'compression_pct': compression,
                           'summary': summary})

summary_df = pd.DataFrame(summaries_demo)
print(f'\n📊 Avg Compression: {summary_df["compression_pct"].mean():.1f}%')
print('✅ Module 10A complete — abstractive summarization working')

In [ ]:
# ============================================================
# CELL 9 — MODULE 10B: SEMANTIC SEARCH
# ============================================================
print('[10B] Building semantic search index...')

# Load sentence transformer
sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# Build embeddings for a 500-article index (speed)
INDEX_SIZE = 500
index_df = df.sample(n=INDEX_SIZE, random_state=42).reset_index(drop=True)

print(f'  Encoding {INDEX_SIZE} articles...')
# Use first 256 tokens for speed
short_texts = index_df['text'].apply(lambda t: ' '.join(str(t).split()[:256]))
corpus_embeddings = sem_model.encode(short_texts.tolist(),
                                      convert_to_tensor=True,
                                      show_progress_bar=True)

def semantic_search(query, top_k=5):
    """Find most semantically similar articles to a natural language query."""
    q_emb = sem_model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, corpus_embeddings, top_k=top_k)[0]
    results = []
    for hit in hits:
        row = index_df.iloc[hit['corpus_id']]
        results.append({
            'score': round(hit['score'], 4),
            'category': row['category'],
            'snippet': str(row['text'])[:200] + '...'
        })
    return results

# Demo queries
demo_queries = [
    'Premier League transfer news and player performance',
    'Government policy on economic growth',
    'New smartphone technology release'
]

print('\n🔍 Semantic Search Demo:')
for q in demo_queries:
    print(f'\n  Query: "{q}"')
    results = semantic_search(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f'    {i}. [{r["category"]}] score={r["score"]} — {r["snippet"][:100]}...')

print('\n✅ Module 10B complete — semantic search index ready')

In [ ]:
# ============================================================
# CELL 10 — MODULE 11: MULTILINGUAL INTELLIGENCE
# Language Detection + Translation + Cross-Language Analysis
# ============================================================
print('=' * 60)
print('MODULE 11 — MULTILINGUAL INTELLIGENCE')
print('=' * 60)

# --- 11A: Language Detection on corpus ---
print('\n[11A] Running language detection on corpus sample...')

def detect_lang(text):
    try:
        return detect(str(text)[:300])
    except:
        return 'unknown'

sample_400 = df.sample(400, random_state=42).copy()
sample_400['detected_lang'] = sample_400['text'].apply(detect_lang)

lang_counts = sample_400['detected_lang'].value_counts()
print(f'\n  Languages detected in 400-article sample:')
print(lang_counts.to_string())

# --- 11B: Translation — foreign articles to English ---
print('\n[11B] Translation Integration Demo...')

# Test articles in multiple languages (news-style)
foreign_samples = [
    {'lang': 'es', 'text': 'El gobierno anunció nuevas medidas económicas para combatir la inflación creciente en el país. Los expertos coinciden en que estas políticas podrían afectar a los ciudadanos más vulnerables.'},
    {'lang': 'fr', 'text': "L'équipe nationale a remporté le championnat après une performance exceptionnelle lors de la finale. Les supporters ont célébré la victoire dans tout le pays."},
    {'lang': 'de', 'text': 'Das Technologieunternehmen kündigte einen neuen Quantencomputer an, der bisher ungelöste Probleme in der Wissenschaft lösen könnte. Dies markiert einen Meilenstein in der Forschung.'},
    {'lang': 'pt', 'text': 'Os líderes mundiais se reuniram para discutir mudanças climáticas e acordos de energia renovável. A cúpula foi considerada um avanço significativo nas negociações internacionais.'},
    {'lang': 'zh-CN', 'text': '人工智能技术正在迅速发展，改变着各行各业。研究人员表示，这将对未来的劳动力市场产生深远影响。'}
]

print('\n  Translation Results:')
translation_results = []
for sample in foreign_samples:
    try:
        translated = GoogleTranslator(source='auto', target='en').translate(sample['text'])
        # Classify the translated text
        clean_trans = preprocess(translated)
        vec = tfidf.transform([clean_trans])
        pred_cat = lr.predict(vec)[0]
        sentiment = sid.polarity_scores(translated)['compound']
        translation_results.append({
            'source_lang': sample['lang'],
            'translated': translated[:150],
            'predicted_category': pred_cat,
            'sentiment': round(sentiment, 3)
        })
        print(f'\n  [{sample["lang"]}] → Category: {pred_cat} | Sentiment: {sentiment:.3f}')
        print(f'    Translation: {translated[:120]}...')
    except Exception as e:
        print(f'  [{sample["lang"]}] Translation error: {e}')

trans_df = pd.DataFrame(translation_results)

# --- 11C: Cross-language consistency analysis ---
print('\n[11C] Cross-Language Sentiment Distribution...')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Language detection donut
top_langs = lang_counts.head(6)
axes[0].pie(top_langs.values,
            labels=[f'{l} ({v})' for l,v in zip(top_langs.index, top_langs.values)],
            autopct='%1.1f%%', startangle=90,
            colors=plt.cm.Set2.colors[:len(top_langs)])
axes[0].set_title('Language Detection Distribution\n(400-article sample)', fontweight='bold')

# Translation results sentiment
if len(trans_df) > 0:
    colors_sent = ['#2ca02c' if s >= 0.05 else '#d62728' if s <= -0.05 else '#aec7e8'
                   for s in trans_df['sentiment']]
    bars = axes[1].bar(trans_df['source_lang'], trans_df['sentiment'], color=colors_sent, width=0.6)
    axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_title('Translated Article Sentiment by Source Language', fontweight='bold')
    axes[1].set_xlabel('Source Language')
    axes[1].set_ylabel('VADER Compound Score')
    for bar, cat in zip(bars, trans_df['predicted_category']):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.01, cat,
                     ha='center', va='bottom', fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.savefig('m11_multilingual.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Module 11 complete — language detection, translation, cross-language classification')

In [ ]:
# ============================================================
# CELL 11 — MODULE 12: CONVERSATIONAL INTERFACE
# Full NL query engine with Gradio chat UI
# ============================================================
print('=' * 60)
print('MODULE 12 — CONVERSATIONAL INTERFACE')
print('=' * 60)

# Pre-compute stats for the query engine
stats_cache = {
    'accuracy': acc,
    'total_articles': len(df),
    'category_counts': df['category'].value_counts().to_dict(),
    'avg_sentiment': df.groupby('category')['sent_compound'].mean().round(3).to_dict(),
    'top_topics': {},
    'avg_words': df.groupby('category')['word_count'].mean().round(0).astype(int).to_dict(),
}

# Top 3 words per topic per category
for cat in CATS:
    cat_docs = df[df['category'] == cat]['clean'].tolist()
    cv = CountVectorizer(max_features=20)
    m = cv.fit_transform(cat_docs)
    top_w = sorted(zip(cv.get_feature_names_out(), m.sum(axis=0).A1),
                   key=lambda x: -x[1])[:5]
    stats_cache['top_topics'][cat] = [w for w, _ in top_w]

print('✅ Stats cache built')

# ---- INTENT PARSER + RESPONSE GENERATOR ----
def parse_and_respond(user_input, history):
    q = user_input.lower().strip()
    
    # --- Intent: Classify article ---
    if any(k in q for k in ['classify', 'what category', 'what type', 'predict']):
        # Extract article text after the intent keyword
        for kw in ['classify:', 'classify this:', 'classify this article:', 'classify the following:']:
            if kw in q:
                article_text = user_input[user_input.lower().index(kw)+len(kw):].strip()
                if len(article_text) > 20:
                    clean = preprocess(article_text)
                    vec = tfidf.transform([clean])
                    pred = lr.predict(vec)[0]
                    probs = lr.predict_proba(vec)[0]
                    top3 = sorted(zip(lr.classes_, probs), key=lambda x: -x[1])[:3]
                    sent = sid.polarity_scores(article_text)['compound']
                    resp = f'**Classification Result**\n\n'
                    resp += f'📁 **Category:** {pred.upper()}\n'
                    resp += f'📊 **Confidence scores:**\n'
                    for cls, prob in top3:
                        bar = '█' * int(prob * 20)
                        resp += f'  {cls:15s} {bar} {prob:.1%}\n'
                    sent_label = 'Positive 😊' if sent >= 0.05 else 'Negative 😞' if sent <= -0.05 else 'Neutral 😐'
                    resp += f'\n💬 **Sentiment:** {sent_label} (score: {sent:.3f})'
                    return resp
        return 'Please format as: **classify:** [paste your article text here]'

    # --- Intent: Summarize ---
    elif any(k in q for k in ['summarize', 'summarise', 'summary', 'tldr', 'tl;dr']):
        for kw in ['summarize:', 'summarise:', 'summary:', 'tldr:', 'tl;dr:']:
            if kw in q:
                article_text = user_input[user_input.lower().index(kw)+len(kw):].strip()
                if len(article_text.split()) >= 40:
                    s = safe_summarize(article_text)
                    orig_w = len(article_text.split())
                    summ_w = len(s.split())
                    comp = round((1 - summ_w/orig_w)*100, 1)
                    return f'**Summary** ({orig_w} → {summ_w} words, {comp}% compression)\n\n{s}'
                else:
                    return 'Please provide at least 40 words for summarization.'
        return 'Format: **summarize:** [paste article text]'

    # --- Intent: Semantic search ---
    elif any(k in q for k in ['search', 'find articles', 'find news', 'related to', 'about']):
        for kw in ['search:', 'find articles about:', 'find:', 'search for:']:
            if kw in q:
                query_text = user_input[user_input.lower().index(kw)+len(kw):].strip()
                if len(query_text) > 3:
                    results = semantic_search(query_text, top_k=4)
                    resp = f'**Semantic Search Results** for: "{query_text}"\n\n'
                    for i, r in enumerate(results, 1):
                        resp += f'**{i}.** [{r["category"].upper()}] — similarity: {r["score"]}\n'
                        resp += f'{r["snippet"][:180]}...\n\n'
                    return resp
        # fallback — treat whole query as search
        results = semantic_search(user_input, top_k=3)
        resp = f'**Top 3 semantically similar articles:**\n\n'
        for i, r in enumerate(results, 1):
            resp += f'**{i}.** [{r["category"].upper()}] score={r["score"]}\n{r["snippet"][:150]}...\n\n'
        return resp

    # --- Intent: Statistics / Analytics ---
    elif any(k in q for k in ['stat', 'how many', 'accuracy', 'performance', 'sentiment score',
                               'distribution', 'breakdown', 'count']):
        resp = '**📊 NewsBot 2.0 — System Statistics**\n\n'
        resp += f'- **Total articles indexed:** {stats_cache["total_articles"]:,}\n'
        resp += f'- **Classification accuracy:** {stats_cache["accuracy"]:.1%}\n\n'
        resp += '**Category Breakdown:**\n'
        for cat, cnt in stats_cache['category_counts'].items():
            sent = stats_cache['avg_sentiment'][cat]
            words = stats_cache['avg_words'][cat]
            sent_icon = '📈' if sent >= 0.05 else '📉' if sent <= -0.05 else '➡️'
            resp += f'  - **{cat.title()}:** {cnt} articles | avg sentiment {sent:.3f} {sent_icon} | avg {words} words\n'
        return resp

    # --- Intent: Top topics per category ---
    elif any(k in q for k in ['topic', 'keyword', 'theme', 'key word']):
        resp = '**🔑 Top Keywords by Category:**\n\n'
        for cat, words in stats_cache['top_topics'].items():
            resp += f'**{cat.title()}:** {" · ".join(words)}\n'
        return resp

    # --- Intent: Translate + classify ---
    elif any(k in q for k in ['translate', 'in spanish', 'in french', 'non-english', 'foreign']):
        for kw in ['translate:']:
            if kw in q:
                text_to_trans = user_input[user_input.lower().index(kw)+len(kw):].strip()
                try:
                    lang = detect(text_to_trans)
                    translated = GoogleTranslator(source='auto', target='en').translate(text_to_trans)
                    clean = preprocess(translated)
                    pred = lr.predict(tfidf.transform([clean]))[0]
                    sent = sid.polarity_scores(translated)['compound']
                    return (f'**🌐 Translation Result**\n\n'
                            f'- **Detected language:** {lang}\n'
                            f'- **English translation:** {translated[:300]}\n'
                            f'- **Predicted category:** {pred.upper()}\n'
                            f'- **Sentiment:** {sent:.3f}')
                except Exception as e:
                    return f'Translation error: {e}'
        return 'Format: **translate:** [non-English text]'

    # --- Intent: Help ---
    elif any(k in q for k in ['help', 'what can you', 'commands', 'how do i', 'guide']):
        return ('**🤖 NewsBot 2.0 — What I can do:**\n\n'
                '1. **classify:** [article text] → Predicts category + sentiment\n'
                '2. **summarize:** [article text] → Abstractive summary\n'
                '3. **search:** [topic or question] → Finds semantically similar articles\n'
                '4. **translate:** [foreign text] → Translates + classifies\n'
                '5. **stats** → System performance & category breakdown\n'
                '6. **topics** → Top keywords per category\n\n'
                'Try: `classify: The Prime Minister announced new policies today...`')

    # --- Fallback: semantic search ---
    else:
        results = semantic_search(user_input, top_k=2)
        resp = f'I found some related articles. Type **help** to see all commands.\n\n'
        for i, r in enumerate(results, 1):
            resp += f'**{i}.** [{r["category"].upper()}] {r["snippet"][:150]}...\n\n'
        return resp

print('✅ Conversational engine ready')

In [ ]:
# ============================================================
# CELL 12 — LAUNCH GRADIO CONVERSATIONAL INTERFACE
# ============================================================

THEME_CSS = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.chatbot .message { border-radius: 12px !important; }
"""

with gr.Blocks(title='NewsBot Intelligence Engine 2.0', css=THEME_CSS,
               theme=gr.themes.Soft(primary_hue='blue')) as demo:
    
    gr.Markdown("""
    # 🤖 NewsBot Intelligence Engine 2.0
    **ITAI 2373 Final Project | Leroy Brown | Southern Shade Technologies**
    
    Ask me to **classify**, **summarize**, **search**, **translate**, or request **stats**. Type `help` to start.
    """)
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=480, label='NewsBot 2.0 Chat',
                                 bubble_full_width=False)
            with gr.Row():
                user_input = gr.Textbox(placeholder='Type a command or question...',
                                        label='', scale=5, container=False)
                send_btn = gr.Button('Send →', variant='primary', scale=1)
            
            gr.Examples(
                examples=[
                    ['stats'],
                    ['topics'],
                    ['help'],
                    ['search: artificial intelligence and machine learning'],
                    ['classify: The Prime Minister announced new fiscal policy measures today aimed at reducing the deficit.'],
                    ['translate: El gobierno anunció nuevas medidas económicas para combatir la inflación.'],
                ],
                inputs=user_input,
                label='Quick examples (click to try)'
            )
        
        with gr.Column(scale=1):
            gr.Markdown("""
            ### 📌 Command Reference
            | Command | Usage |
            |---------|-------|
            | `classify:` | Paste article text |
            | `summarize:` | Paste article text |
            | `search:` | Enter topic/query |
            | `translate:` | Foreign language text |
            | `stats` | View system metrics |
            | `topics` | Keywords by category |
            | `help` | Full guide |
            
            ### 🏆 System Performance
            - **Accuracy:** 97.6%
            - **Articles:** 2,000 BBC
            - **Categories:** 5
            - **Languages:** 5+ supported
            """)
    
    def respond(message, history):
        if not message.strip():
            return history, ''
        reply = parse_and_respond(message, history)
        history.append((message, reply))
        return history, ''
    
    send_btn.click(respond, [user_input, chatbot], [chatbot, user_input])
    user_input.submit(respond, [user_input, chatbot], [chatbot, user_input])

demo.launch(share=True, quiet=False)

---
## 📊 System Summary — NewsBot 2.0

| Module | Technique | Key Result |
|--------|-----------|------------|
| M1–M8 | Full midterm pipeline | 97.6% classification accuracy |
| **M9** | LDA (10 topics) + Multi-dim VADER | Topics extracted per category |
| **M10** | DistilBART + SentenceTransformers | ~60% compression; semantic search |
| **M11** | langdetect + deep-translator | 5+ language auto-classify pipeline |
| **M12** | Gradio conversational UI | 7 intent types, live public URL |

### Author
**Leroy Brown** | Founder & CTO, Southern Shade Technologies  
*ITAI 2373 — NLP | Houston Community College*